In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
train_path = "Subtask_A_train.csv"
test_path  = "Subtask_A_test_noLabel.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print(train_df.head())
print(test_df.head())

   id                                               text          label
0   1          against genocide that the Zionists commit  Pro-Palestine
1   2  I am not the devil to stand with Israel. becau...     Pro-Israel
2   3  Nope!! Hamas, Hezbollah, and Iran are all terr...     Pro-Israel
3   4  The president has the powers, but he can be re...        Neutral
4   5  He can order whichever book he wants. However,...  Pro-Palestine
     id                                               text  prediction
0  1191       US is occupied by Israel just like Palestine         NaN
1  1192  Israel war in Gaza is a n action of self defen...         NaN
2  1193  The two faces of kamala Harris. One face says,...         NaN
3  1194  as for jahiliya ? I'm not a jew nor a zionist....         NaN
4  1195  Free Palestine Forever. World must Stop Trade ...         NaN


In [ ]:
label2id = {
    "Pro-Palestine": 0,
    "Pro-Israel": 1,
    "Neutral": 2
}

id2label = {v: k for k, v in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df[["text", "label_id"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]]
)

In [ ]:
MODEL_NAME = "xlm-roberta-base"
# For Arabic-heavy data, try:
# MODEL_NAME = "UBC-NLP/MARBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
train_dataset.set_format("torch")
test_dataset.set_format("torch")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/980 [00:00<?, ? examples/s]

Map:   0%|          | 0/211 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="./stance_model",
    eval_strategy="no",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    fp16=True,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()

Step,Training Loss
50,1.114478
100,1.034184
150,0.773656


TrainOutput(global_step=186, training_loss=0.9095912851313109, metrics={'train_runtime': 48.9526, 'train_samples_per_second': 60.058, 'train_steps_per_second': 3.8, 'total_flos': 386776724060160.0, 'train_loss': 0.9095912851313109, 'epoch': 3.0})

In [ ]:
predictions = trainer.predict(test_dataset)
pred_ids = np.argmax(predictions.predictions, axis=1)

pred_labels = [id2label[i] for i in pred_ids]

In [ ]:
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "prediction": pred_labels
})

submission_df.to_csv("prediction.csv", index=False, encoding="utf-8")
submission_df.head()

,id,prediction
0,1191,Pro-Palestine
1,1192,Neutral
2,1193,Neutral
3,1194,Pro-Palestine
4,1195,Pro-Palestine


In [ ]:
import zipfile

with zipfile.ZipFile("prediction2.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction.csv")

# Experiment 02 (The Perfect Run)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import zipfile

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

In [ ]:
train_path = "Subtask_A_train.csv"
test_path  = "Subtask_A_test_noLabel.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

train_df.head()
test_df.head()

,id,text,prediction
0,1191,US is occupied by Israel just like Palestine,NaN
1,1192,Israel war in Gaza is a n action of self defen...,NaN
2,1193,"The two faces of kamala Harris. One face says,...",NaN
3,1194,as for jahiliya ? I'm not a jew nor a zionist....,NaN
4,1195,Free Palestine Forever. World must Stop Trade ...,NaN


In [ ]:
label2id = {
    "Pro-Palestine": 0,
    "Pro-Israel": 1,
    "Neutral": 2
}
id2label = {v: k for k, v in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["label_id"].values
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)

Class weights: tensor([1.0020, 0.9990, 0.9990])


In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
def tokenize_function(tokenizer, examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

In [ ]:
from transformers import EarlyStoppingCallback

In [ ]:
def train_cv_model(model_name, train_df, test_df, n_splits=5):

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    test_probs = np.zeros((len(test_df), 3))
    all_val_preds = []
    all_val_labels = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(train_df, train_df["label_id"])
    ):
        print(f"\n🔹 Training {model_name} | Fold {fold+1}/{n_splits}")

        train_fold = train_df.iloc[train_idx]
        val_fold   = train_df.iloc[val_idx]

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        train_ds = Dataset.from_pandas(
            train_fold[["text", "label_id"]]
        ).rename_column("label_id", "labels")

        val_ds = Dataset.from_pandas(
            val_fold[["text", "label_id"]]
        ).rename_column("label_id", "labels")

        test_ds = Dataset.from_pandas(
            test_df[["text"]]
        )

        train_ds = train_ds.map(
            lambda x: tokenize_function(tokenizer, x),
            batched=True
        )
        val_ds = val_ds.map(
            lambda x: tokenize_function(tokenizer, x),
            batched=True
        )
        test_ds = test_ds.map(
            lambda x: tokenize_function(tokenizer, x),
            batched=True
        )

        train_ds.set_format("torch")
        val_ds.set_format("torch")
        test_ds.set_format("torch")

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=3,
            id2label=id2label,
            label2id=label2id
        )

        args = TrainingArguments(
            output_dir=f"./{model_name.replace('/', '_')}_fold{fold}",
            eval_strategy="epoch",
            save_strategy="no",
            learning_rate=4e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=8,
            weight_decay=0.01,
            fp16=True,
            logging_steps=50,
            report_to="none"
        )

        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            class_weights=class_weights
        )

        trainer.train()

        # 🔹 Validation metrics
        val_logits = trainer.predict(val_ds).predictions
        val_preds = np.argmax(val_logits, axis=1)

        all_val_preds.extend(val_preds)
        all_val_labels.extend(val_fold["label_id"].values)

        acc = accuracy_score(val_fold["label_id"], val_preds)
        p, r, f1, _ = precision_recall_fscore_support(
            val_fold["label_id"],
            val_preds,
            average="macro"
        )

        print(f"Fold {fold+1} | Acc: {acc:.4f} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")

        # 🔹 Test predictions
        test_logits = trainer.predict(test_ds).predictions
        test_probs += torch.softmax(
            torch.tensor(test_logits), dim=1
        ).numpy()

    print("\n===== Cross-Validation Overall Metrics =====")
    print(
        classification_report(
            all_val_labels,
            all_val_preds,
            target_names=[id2label[i] for i in range(3)]
        )
    )

    return test_probs / n_splits

In [ ]:
deberta_probs = train_cv_model(
    "UBC-NLP/MARBERT",
    train_df,
    test_df
)


🔹 Training UBC-NLP/MARBERT | Fold 1/5


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

Epoch,Training Loss,Validation Loss
1,No log,0.507431
2,0.897023,0.497798
3,0.578493,0.442895
4,0.340723,0.458744
5,0.195283,0.511355
6,0.128828,0.799868
7,0.052041,0.661761
8,0.035966,0.648172


Fold 5 | Acc: 0.8878 | P: 0.8954 | R: 0.8884 | F1: 0.8856



===== Cross-Validation Overall Metrics =====
               precision    recall  f1-score   support

Pro-Palestine       0.86      0.91      0.88       326
   Pro-Israel       0.87      0.94      0.90       327
      Neutral       0.92      0.78      0.84       327

     accuracy                           0.88       980
    macro avg       0.88      0.88      0.88       980
 weighted avg       0.88      0.88      0.88       980



In [ ]:
ensemble_probs = deberta_probs
final_preds = np.argmax(ensemble_probs, axis=1)
final_labels = [id2label[i] for i in final_preds]

In [ ]:
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "prediction": final_labels
})

submission_df.to_csv(
    "prediction.csv",
    index=False,
    encoding="utf-8"
)

submission_df.head()

,id,prediction
0,1191,Neutral
1,1192,Pro-Israel
2,1193,Pro-Palestine
3,1194,Pro-Palestine
4,1195,Pro-Palestine


In [ ]:
with zipfile.ZipFile("prediction9.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction.csv")

print("prediction.zip ready ")

prediction.zip ready ✅


# Experiment 03

In [ ]:
# ============================================
# Actor-Level Stance Detection
# MARBERT + ARBERT Ensemble (Weighted + Stacking)
# ============================================

import os
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression

# ======================
# Paths
# ======================
TRAIN_PATH = "Subtask_A_train.csv"
TEST_PATH  = "Subtask_A_val_noLabel.csv"

# ======================
# Load data
# ======================
train_df = pd.read_csv(TRAIN_PATH).dropna(subset=["text", "label"]).reset_index(drop=True)
test_df  = pd.read_csv(TEST_PATH)

# ======================
# Label mapping
# ======================
label_encoder = LabelEncoder()
train_df["label_id"] = label_encoder.fit_transform(train_df["label"])
id2label = {i: l for i, l in enumerate(label_encoder.classes_)}
label2id = {l: i for i, l in id2label.items()}
NUM_LABELS = len(id2label)
print("Label mapping:", label2id)

# ======================
# Class weights
# ======================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label_id"]),
    y=train_df["label_id"]
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# ======================
# Tokenization
# ======================
def tokenize_function(tokenizer, examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=192  # increased from 128
    )

# ======================
# Weighted Trainer
# ======================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ======================
# CV training function
# ======================
def train_cv_model(model_name, train_df, test_df, n_splits=5):

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    test_probs = np.zeros((len(test_df), NUM_LABELS))
    all_val_preds, all_val_labels = [], []
    val_fold_probs_list = []

    fold_f1_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label_id"])):
        print(f"\n🔹 Training {model_name} | Fold {fold+1}/{n_splits}")

        train_fold = train_df.iloc[train_idx]
        val_fold   = train_df.iloc[val_idx]

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        train_ds = Dataset.from_pandas(train_fold[["text", "label_id"]]).rename_column("label_id", "labels")
        val_ds   = Dataset.from_pandas(val_fold[["text", "label_id"]]).rename_column("label_id", "labels")
        test_ds  = Dataset.from_pandas(test_df[["text"]])

        train_ds = train_ds.map(lambda x: tokenize_function(tokenizer, x), batched=True)
        val_ds   = val_ds.map(lambda x: tokenize_function(tokenizer, x), batched=True)
        test_ds  = test_ds.map(lambda x: tokenize_function(tokenizer, x), batched=True)

        train_ds.set_format("torch")
        val_ds.set_format("torch")
        test_ds.set_format("torch")

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=NUM_LABELS,
            id2label=id2label,
            label2id=label2id
        )

        args = TrainingArguments(
            output_dir=f"./{model_name.replace('/', '_')}_fold{fold}",
            eval_strategy="epoch",
            save_strategy="no",
            learning_rate=3e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=4,
            weight_decay=0.01,
            fp16=True,
            logging_steps=50,
            report_to="none"
        )

        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            class_weights=class_weights
        )

        trainer.train()

        # Validation predictions
        val_logits = trainer.predict(val_ds).predictions
        val_probs = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
        val_preds = np.argmax(val_probs, axis=1)

        all_val_preds.extend(val_preds)
        all_val_labels.extend(val_fold["label_id"].values)
        val_fold_probs_list.append(val_probs)

        # Compute F1 for fold
        _, _, f1, _ = precision_recall_fscore_support(val_fold["label_id"], val_preds, average="macro")
        fold_f1_scores.append(f1)

        acc = accuracy_score(val_fold["label_id"], val_preds)
        print(f"Fold {fold+1} | Acc {acc:.4f} | F1 {f1:.4f}")

        # Test predictions
        test_logits = trainer.predict(test_ds).predictions
        test_probs += torch.softmax(torch.tensor(test_logits), dim=1).numpy()

    print("\n===== CV Metrics =====")
    print(classification_report(all_val_labels, all_val_preds, target_names=id2label.values()))

    avg_f1 = np.mean(fold_f1_scores)
    print(f"Average CV Macro F1: {avg_f1:.4f}")

    return test_probs / n_splits, val_fold_probs_list, fold_f1_scores

# ======================
# Train ARBERT and MARBERT
# ======================
arbert_probs, arbert_val_probs, arbert_f1s = train_cv_model("UBC-NLP/ARBERT", train_df, test_df)
marbert_probs, marbert_val_probs, marbert_f1s = train_cv_model("UBC-NLP/MARBERT", train_df, test_df)

# ======================
# Weighted ensemble
# ======================
# Weight by fold F1
arbert_weight = np.mean(arbert_f1s)
marbert_weight = np.mean(marbert_f1s)
total_weight = arbert_weight + marbert_weight

ensemble_probs = (marbert_weight * marbert_probs + arbert_weight * arbert_probs) / total_weight
weighted_preds = np.argmax(ensemble_probs, axis=1)
weighted_labels = [id2label[i] for i in weighted_preds]

# ======================
# Stacking ensemble
# ======================
# Prepare meta-features using fold validation probs
val_meta_features = np.hstack([
    np.vstack(marbert_val_probs),
    np.vstack(arbert_val_probs)
])
val_meta_labels = np.array(train_df["label_id"])

# Logistic Regression as meta-learner
stacker = LogisticRegression(max_iter=500)
stacker.fit(val_meta_features, val_meta_labels)

# Prepare test meta-features
test_meta_features = np.hstack([marbert_probs, arbert_probs])
stack_preds = stacker.predict(test_meta_features)
stack_labels = [id2label[i] for i in stack_preds]

# ======================
# Save submissions
# ======================
# Weighted
submission_weighted = pd.DataFrame({
    "id": test_df["id"],
    "prediction": weighted_labels
})
submission_weighted.to_csv("prediction_weighted.csv", index=False)

# Stacked
submission_stacked = pd.DataFrame({
    "id": test_df["id"],
    "prediction": stack_labels
})
submission_stacked.to_csv("prediction_stacked.csv", index=False)

# Zip files
with zipfile.ZipFile("prediction_weighted.zip", "w") as z:
    z.write("prediction_weighted.csv")
with zipfile.ZipFile("prediction_stacked.zip", "w") as z:
    z.write("prediction_stacked.csv")

print("\n✅ DONE — Weighted & Stacked submissions ready")

Label mapping: {'Neutral': 0, 'Pro-Israel': 1, 'Pro-Palestine': 2}

🔹 Training UBC-NLP/ARBERT | Fold 1/5


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/374 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

Epoch,Training Loss,Validation Loss
1,No log,0.635588
2,0.964369,0.443342
3,0.523277,0.426692
4,0.280304,0.412984


Fold 5 | Acc 0.8724 | F1 0.8708



===== CV Metrics =====
               precision    recall  f1-score   support

      Neutral       0.88      0.77      0.82       327
   Pro-Israel       0.83      0.94      0.89       327
Pro-Palestine       0.84      0.83      0.84       326

     accuracy                           0.85       980
    macro avg       0.85      0.85      0.85       980
 weighted avg       0.85      0.85      0.85       980

Average CV Macro F1: 0.8482

✅ DONE — Weighted & Stacked submissions ready


# Hopeless Trial

Trial 01 Folder

In [ ]:
# ==========================================================
# FULL TRAINING + ENSEMBLE + ZIP EXPORT SCRIPT
# ==========================================================

!pip install -q transformers datasets accelerate scikit-learn

import os
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ==========================================================
# 1️ LOAD DATA
# ==========================================================

train_df = pd.read_csv("Subtask_A_train.csv")
val_df   = pd.read_csv("Subtask_A_val_labeled.csv")
test_df  = pd.read_csv("Subtask_A_test_noLabel.csv")

train_df = pd.concat([train_df, val_df], ignore_index=True)
train_df = train_df.dropna(subset=["text", "label"]).reset_index(drop=True)

# ==========================================================
# 2️ ENCODE LABELS
# ==========================================================

le = LabelEncoder()
train_df["label_id"] = le.fit_transform(train_df["label"])

id2label = {i: l for i, l in enumerate(le.classes_)}
label2id = {l: i for i, l in id2label.items()}
NUM_LABELS = len(id2label)

# ==========================================================
# 3 CLASS WEIGHTS
# ==========================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label_id"]),
    y=train_df["label_id"]
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# ==========================================================
# 4️ TOKENIZER
# ==========================================================

def tokenize(tokenizer, examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

# ==========================================================
# 5️ WEIGHTED TRAINER
# ==========================================================

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ==========================================================
# 6️ TRAIN FUNCTION
# ==========================================================

def train_model(model_name, epochs, lr):

    print(f"\n==============================")
    print(f"Training {model_name}")
    print(f"==============================")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    test_probs = np.zeros((len(test_df), NUM_LABELS))
    fold_f1_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(train_df, train_df["label_id"])):

        print(f"\n--- Fold {fold+1} ---")

        train_fold = train_df.iloc[tr_idx]
        val_fold   = train_df.iloc[val_idx]

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        train_ds = Dataset.from_pandas(train_fold[["text","label_id"]]).rename_column("label_id","labels")
        val_ds   = Dataset.from_pandas(val_fold[["text","label_id"]]).rename_column("label_id","labels")
        test_ds  = Dataset.from_pandas(test_df[["text"]])

        train_ds = train_ds.map(lambda x: tokenize(tokenizer,x), batched=True)
        val_ds   = val_ds.map(lambda x: tokenize(tokenizer,x), batched=True)
        test_ds  = test_ds.map(lambda x: tokenize(tokenizer,x), batched=True)

        train_ds.set_format("torch")
        val_ds.set_format("torch")
        test_ds.set_format("torch")

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=NUM_LABELS,
            id2label=id2label,
            label2id=label2id
        )

        args = TrainingArguments(
            output_dir=f"./tmp_{fold}",
            learning_rate=lr,
            num_train_epochs=epochs,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            eval_strategy="epoch",
            save_strategy="no",
            fp16=True,
            logging_steps=100,
            report_to="none"
        )

        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            class_weights=class_weights
        )

        trainer.train()

        # Validation F1
        val_logits = trainer.predict(val_ds).predictions
        val_preds = np.argmax(val_logits, axis=1)

        _,_,f1,_ = precision_recall_fscore_support(
            val_fold["label_id"],
            val_preds,
            average="macro"
        )

        fold_f1_scores.append(f1)
        print(f"Fold {fold+1} Macro-F1: {f1:.4f}")

        # Test prediction
        test_logits = trainer.predict(test_ds).predictions
        test_probs += torch.softmax(torch.tensor(test_logits), dim=1).numpy()

    avg_f1 = np.mean(fold_f1_scores)
    print("\nAverage Macro-F1:", avg_f1)

    return test_probs/5, avg_f1, fold_f1_scores

# ==========================================================
# TRAIN MODELS
# ==========================================================

arbert_probs, arbert_f1, arbert_folds = train_model("UBC-NLP/ARBERT", epochs=5, lr=5e-5)
marbert_probs, marbert_f1, marbert_folds = train_model("UBC-NLP/MARBERT", epochs=5, lr=5e-5)

# ==========================================================
SAVE INDIVIDUAL SUBMISSIONS
# ==========================================================

def save_submission(probs, name):
    preds = np.argmax(probs, axis=1)
    labels = [id2label[i] for i in preds]

    sub = pd.DataFrame({
        "id": test_df["id"],
        "prediction": labels
    })

    csv_name = f"{name}.csv"
    zip_name = f"{name}.zip"

    sub.to_csv(csv_name, index=False)

    with zipfile.ZipFile(zip_name, "w") as z:
        z.write(csv_name)

    print(f"Saved {zip_name}")

save_submission(arbert_probs, "ARBERT_submission")
save_submission(marbert_probs, "MARBERT_submission")

# ==========================================================
#SAVE CV RESULTS
# ==========================================================

cv_results = pd.DataFrame({
    "Model": ["ARBERT", "MARBERT"],
    "Average_F1": [arbert_f1, marbert_f1]
})

cv_results.to_csv("CV_results.csv", index=False)

# ==========================================================
# FINAL ZIP (ALL RESULTS)
# ==========================================================

with zipfile.ZipFile("ALL_RESULTS.zip", "w") as z:
    z.write("ARBERT_submission.zip")
    z.write("MARBERT_submission.zip")
    z.write("CV_results.csv")

print("\nALL_RESULTS.zip READY")

Using device: cuda

Training UBC-NLP/ARBERT

--- Fold 1 ---


Map:   0%|          | 0/784 [00:00<?, ? examples/s]

Map:   0%|          | 0/196 [00:00<?, ? examples/s]

Map:   0%|          | 0/211 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

Epoch,Training Loss,Validation Loss
1,No log,0.981234
2,1.068220,0.749204
3,0.791847,0.721290
4,0.524460,0.717154
5,0.381751,0.699174


Fold 5 Macro-F1: 0.8513



Average Macro-F1: 0.7726257336677615
Saved ARBERT_submission.zip
Saved MARBERT_submission.zip

🎯 ALL_RESULTS.zip READY


Trail 02 Folder

# Good Experiment

In [ ]:
# ============================================
# Actor-Level Stance Detection
# MARBERT + ARBERT Ensemble (Weighted + Stacking)
# ============================================

import os
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression

# ======================
# UPDATED Paths
# ======================
TRAIN_PATH = "Subtask_A_train.csv"
TEST_PATH  = "Subtask_A_test_noLabel.csv"

# ======================
# Load data
# ======================
train_df = pd.read_csv(TRAIN_PATH).dropna(subset=["text", "label"]).reset_index(drop=True)
test_df  = pd.read_csv(TEST_PATH).dropna(subset=["text"]).reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# ======================
# Label mapping
# ======================
label_encoder = LabelEncoder()
train_df["label_id"] = label_encoder.fit_transform(train_df["label"])

id2label = {i: l for i, l in enumerate(label_encoder.classes_)}
label2id = {l: i for i, l in id2label.items()}
NUM_LABELS = len(id2label)

print("Label mapping:", label2id)

# ======================
# Class weights
# ======================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label_id"]),
    y=train_df["label_id"]
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# ======================
# Tokenization
# ======================
def tokenize_function(tokenizer, examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=192
    )

# ======================
# Weighted Trainer
# ======================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ======================
# CV training function
# ======================
def train_cv_model(model_name, train_df, test_df, n_splits=5):

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    test_probs = np.zeros((len(test_df), NUM_LABELS))
    all_val_preds, all_val_labels = [], []
    val_fold_probs_list = []
    fold_f1_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label_id"])):
        print(f"\n🔹 Training {model_name} | Fold {fold+1}/{n_splits}")

        train_fold = train_df.iloc[train_idx]
        val_fold   = train_df.iloc[val_idx]

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        train_ds = Dataset.from_pandas(train_fold[["text", "label_id"]]).rename_column("label_id", "labels")
        val_ds   = Dataset.from_pandas(val_fold[["text", "label_id"]]).rename_column("label_id", "labels")
        test_ds  = Dataset.from_pandas(test_df[["text"]])

        train_ds = train_ds.map(lambda x: tokenize_function(tokenizer, x), batched=True)
        val_ds   = val_ds.map(lambda x: tokenize_function(tokenizer, x), batched=True)
        test_ds  = test_ds.map(lambda x: tokenize_function(tokenizer, x), batched=True)

        train_ds.set_format("torch")
        val_ds.set_format("torch")
        test_ds.set_format("torch")

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=NUM_LABELS,
            id2label=id2label,
            label2id=label2id
        )

        args = TrainingArguments(
        output_dir=f"./{model_name.replace('/', '_')}_fold{fold}",
        do_eval=True,
        learning_rate=3e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=4,
        weight_decay=0.01,
        fp16=True,
        logging_steps=50
)

        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            class_weights=class_weights
        )

        trainer.train()

        # Validation predictions
        val_logits = trainer.predict(val_ds).predictions
        val_probs = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
        val_preds = np.argmax(val_probs, axis=1)

        all_val_preds.extend(val_preds)
        all_val_labels.extend(val_fold["label_id"].values)
        val_fold_probs_list.append(val_probs)

        _, _, f1, _ = precision_recall_fscore_support(
            val_fold["label_id"], val_preds, average="macro"
        )
        fold_f1_scores.append(f1)

        print(f"Fold {fold+1} | F1 {f1:.4f}")

        # Test predictions
        test_logits = trainer.predict(test_ds).predictions
        test_probs += torch.softmax(torch.tensor(test_logits), dim=1).numpy()

    print("\n===== CV Metrics =====")
    print(classification_report(all_val_labels, all_val_preds, target_names=id2label.values()))

    avg_f1 = np.mean(fold_f1_scores)
    print(f"Average CV Macro F1: {avg_f1:.4f}")

    return test_probs / n_splits, val_fold_probs_list, fold_f1_scores

# ======================
# Train ARBERT & MARBERT
# ======================
arbert_probs, arbert_val_probs, arbert_f1s = train_cv_model("UBC-NLP/ARBERT", train_df, test_df)
marbert_probs, marbert_val_probs, marbert_f1s = train_cv_model("UBC-NLP/MARBERT", train_df, test_df)

# ======================
# Weighted Ensemble
# ======================
arbert_weight = np.mean(arbert_f1s)
marbert_weight = np.mean(marbert_f1s)
total_weight = arbert_weight + marbert_weight

ensemble_probs = (marbert_weight * marbert_probs + arbert_weight * arbert_probs) / total_weight
weighted_preds = np.argmax(ensemble_probs, axis=1)
weighted_labels = [id2label[i] for i in weighted_preds]

# ======================
# Stacking Ensemble
# ======================
val_meta_features = np.hstack([
    np.vstack(marbert_val_probs),
    np.vstack(arbert_val_probs)
])
val_meta_labels = train_df["label_id"].values

stacker = LogisticRegression(max_iter=500)
stacker.fit(val_meta_features, val_meta_labels)

test_meta_features = np.hstack([marbert_probs, arbert_probs])
stack_preds = stacker.predict(test_meta_features)
stack_labels = [id2label[i] for i in stack_preds]

# ======================
# Save submissions
# ======================
submission_weighted = pd.DataFrame({
    "id": test_df["id"],
    "prediction": weighted_labels
})
submission_weighted.to_csv("prediction_weighted.csv", index=False)

submission_stacked = pd.DataFrame({
    "id": test_df["id"],
    "prediction": stack_labels
})
submission_stacked.to_csv("prediction_stacked.csv", index=False)

with zipfile.ZipFile("prediction_weighted.zip", "w") as z:
    z.write("prediction_weighted.csv")

with zipfile.ZipFile("prediction_stacked.zip", "w") as z:
    z.write("prediction_stacked.csv")

print("\n✅ DONE — Updated for new test file")

Train shape: (980, 3)
Test shape: (211, 3)
Label mapping: {'Neutral': 0, 'Pro-Israel': 1, 'Pro-Palestine': 2}

🔹 Training UBC-NLP/ARBERT | Fold 1/5


Map:   0%|          | 0/784 [00:00<?, ? examples/s]

Map:   0%|          | 0/196 [00:00<?, ? examples/s]

Map:   0%|          | 0/211 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

Step,Training Loss
50,0.977566
100,0.565016
150,0.336535


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 5 | F1 0.8713



===== CV Metrics =====
               precision    recall  f1-score   support

      Neutral       0.90      0.76      0.82       327
   Pro-Israel       0.85      0.94      0.89       327
Pro-Palestine       0.84      0.89      0.87       326

     accuracy                           0.86       980
    macro avg       0.87      0.86      0.86       980
 weighted avg       0.87      0.86      0.86       980

Average CV Macro F1: 0.8602

✅ DONE — Updated for new test file
